[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/06_linear_regression/90_exercises.ipynb)

# Bài tập — Chương 06: Linear Regression (L0 → L7)

Các assignment 91–93 (từ khoá AIVN) luyện cài đặt cơ bản; bộ bài này luyện **hiểu sâu, debug và phân tích**.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.checks import check_close, check_equal, check_less, check_true
from ml_course.utils import numerical_gradient, relative_error, repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết công thức: hàm giả thuyết, cost function MSE (dạng vector), gradient của cost, normal equation, cost của Ridge.

✍️ Trả lời:

**Đáp án:** $h_\theta(X) = X\theta$; $J(\theta) = \frac{1}{2m}(X\theta - y)^\top(X\theta - y)$; $\nabla J = \frac1m X^\top(X\theta - y)$; $\theta = (X^\top X)^{-1}X^\top y$; Ridge: $J + \frac{\lambda}{2m}\sum_{j\ge1}\theta_j^2$.

## L1 — Giải thích bằng lời

**Câu 2.** Vì sao Lasso có thể đưa hệ số về đúng 0 còn Ridge thì không? Giải thích bằng hình học.

✍️ Trả lời:

**Đáp án:** Regularization ⇔ tối ưu MSE trong vùng ràng buộc $\lVert w\rVert \le t$. Vùng L1 là hình thoi có các **góc nằm trên trục**; đường đồng mức elip của MSE thường chạm vùng ràng buộc tại góc → một số hệ số = 0. Vùng L2 là hình tròn trơn, điểm chạm hầu như không nằm đúng trên trục.

## L2 — Tính tay

**Câu 3.** Dữ liệu 3 điểm: $(x, y) = (1, 2), (2, 3), (3, 5)$. Tính tay nghiệm OLS $\theta_1 = \frac{\sum(x - \bar x)(y - \bar y)}{\sum (x - \bar x)^2}$, $\theta_0 = \bar y - \theta_1\bar x$, và $J(\theta) = \frac{1}{2m}\sum(\hat y - y)^2$ tại nghiệm.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
theta1 = 1.5                              # Σ(x−x̄)(y−ȳ) = 3, Σ(x−x̄)² = 2
theta0 = 10 / 3 - 1.5 * 2                 # ȳ = 10/3, x̄ = 2 → θ0 = 1/3
J_opt = ((1 / 6) ** 2 + (1 / 3) ** 2 + (1 / 6) ** 2) / 6     # residual: −1/6, +1/3, −1/6
### END CODE HERE ###

In [ ]:
xs, ys = np.array([1.0, 2, 3]), np.array([2.0, 3, 5])
sk = LinearRegression().fit(xs[:, None], ys)
check_close((theta0, theta1), (sk.intercept_, sk.coef_[0]), message="θ0, θ1")
check_close(J_opt, np.sum((sk.predict(xs[:, None]) - ys) ** 2) / 6, message="J tại nghiệm")

**Câu 4.** Một bước gradient descent từ $\theta = (0, 0)$ với $\alpha = 0.1$ trên dữ liệu câu 3: tính $\theta$ mới.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
theta0_new = 0.1 * np.mean([2, 3, 5])
theta1_new = 0.1 * np.mean([1 * 2, 2 * 3, 3 * 5])
### END CODE HERE ###

In [ ]:
check_close((theta0_new, theta1_new), (0.1 * ys.mean(), 0.1 * np.mean(xs * ys)), message="θ sau 1 bước GD")

## L3 — Code

**Bài 5.** Cài đặt `RidgeGD` (gradient descent, **không phạt bias**, loss $\frac{1}{2m}\lVert X\mathbf w + b - y\rVert^2 + \frac{\lambda}{2m}\lVert\mathbf w\rVert^2$) theo API sklearn với `fit`/`predict`. Kiểm tra: kết quả trùng nghiệm đóng.

In [ ]:
class RidgeGD:
    def __init__(self, lam=1.0, lr=0.1, n_iters=5000):
        self.lam, self.lr, self.n_iters = lam, lr, n_iters

    def fit(self, X, y):
        m, d = X.shape
        self.coef_, self.intercept_ = np.zeros(d), 0.0
        for _ in range(self.n_iters):
            ### START CODE HERE ### (≈ 3 dòng)
            err = X @ self.coef_ + self.intercept_ - y
            self.coef_ -= self.lr / m * (X.T @ err + self.lam * self.coef_)
            self.intercept_ -= self.lr * err.mean()
            ### END CODE HERE ###
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

In [ ]:
Xr = StandardScaler().fit_transform(rng.normal(size=(100, 4)))
yr = Xr @ np.array([2.0, -1.0, 0.5, 0.0]) + 3 + rng.normal(0, 0.5, 100)
ours = RidgeGD(lam=10.0).fit(Xr, yr)
ref = Ridge(alpha=10.0).fit(Xr, yr)             # sklearn: ||y - Xw||² + alpha ||w||² ⇔ cùng nghiệm với λ = alpha
check_close(ours.coef_, ref.coef_, tol=1e-4, message="hệ số khớp sklearn Ridge")
check_close(ours.intercept_, ref.intercept_, tol=1e-4, message="bias không bị phạt")

**Bài 6.** Viết `vif(X)` trả về mảng VIF cho từng cột của `X` (numpy, shape `(n, d)`): $\text{VIF}_j = 1/(1 - R^2_j)$ khi hồi quy cột $j$ theo các cột còn lại.

In [ ]:
def vif(X):
    out = []
    for j in range(X.shape[1]):
        ### START CODE HERE ### (≈ 3 dòng)
        others = np.delete(X, j, axis=1)
        r2 = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
        out.append(1 / (1 - r2))
        ### END CODE HERE ###
    return np.array(out)

In [ ]:
a = rng.normal(size=500)
Xv = np.c_[a, a + rng.normal(0, 0.1, 500), rng.normal(size=500)]
v = vif(Xv)
check_true(v[0] > 50 and v[1] > 50 and v[2] < 1.2, "phát hiện 2 cột gần trùng nhau, cột 3 độc lập")

## L4 — Debug

**Bài 7.** Đồng nghiệp viết GD cho linear regression nhưng cost **tăng vọt** và ra `nan`. Có **hai** lỗi. Tìm và sửa (dùng gradient check nếu cần).

In [ ]:
df = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
Xh = df[["area_m2", "bedrooms", "age_years"]].to_numpy(float)
yh = df["price_billion_vnd"].to_numpy()


def train_buggy_fixed(X, y, lr=0.1, iters=500):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: Xb = np.hstack([np.ones((len(X), 1)), X])        # 🐛 lỗi 1: chưa chuẩn hoá
    # STARTER: theta = np.zeros(Xb.shape[1])
    # STARTER: for _ in range(iters):
    # STARTER:     theta -= lr * Xb.T @ (Xb @ theta - y)        # 🐛 lỗi 2
    # STARTER: return theta, Xb
    Xs = (X - X.mean(0)) / X.std(0)
    Xb = np.hstack([np.ones((len(X), 1)), Xs])
    theta = np.zeros(Xb.shape[1])
    for _ in range(iters):
        theta -= lr / len(y) * Xb.T @ (Xb @ theta - y)
    return theta, Xb
    ### END CODE HERE ###


theta_h, Xb_h = train_buggy_fixed(Xh, yh)
print(theta_h)

In [ ]:
check_true(np.all(np.isfinite(theta_h)), "không còn nan/inf")
check_close(theta_h, np.linalg.lstsq(Xb_h, yh, rcond=None)[0], tol=1e-3, message="hội tụ về nghiệm tối ưu")

**Giải thích:** (1) feature chưa chuẩn hoá (diện tích ~100, tuổi ~15) → bài toán rất méo, lr = 0.1 phân kỳ; (2) gradient thiếu chia cho $m$ → bước nhảy lớn gấp 1 500 lần. Sửa cả hai: chuẩn hoá feature và dùng $\frac{1}{m}X^\top(X\theta - y)$.

## L5 — Phân tích

**Bài 8.** Trên dữ liệu `house_prices.csv` (feature số + one-hot quận), so sánh **MAE trên tập test** (giá gốc, tỷ VND) của 3 cách: (a) Linear Regression trên giá gốc; (b) Linear Regression trên log(giá) rồi `exp` lại, với `log(area_m2)`; (c) như (b) nhưng Ridge(alpha=1). Giải thích kết quả bằng kiến thức bài 08.

In [ ]:
feats = pd.get_dummies(df[["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km", "has_garden", "district"]],
                       columns=["district"], drop_first=True, dtype=float)
X_tr, X_te, y_tr, y_te = train_test_split(feats, df["price_billion_vnd"], test_size=0.25, random_state=0)
from sklearn.metrics import mean_absolute_error

### START CODE HERE ### (≈ 7 dòng)
mae_raw = mean_absolute_error(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
X_tr_l, X_te_l = X_tr.copy(), X_te.copy()
X_tr_l["area_m2"], X_te_l["area_m2"] = np.log(X_tr_l["area_m2"]), np.log(X_te_l["area_m2"])
log_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr_l, np.log(y_tr))
mae_log = mean_absolute_error(y_te, np.exp(log_model.predict(X_te_l)))
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_tr_l, np.log(y_tr))
mae_log_ridge = mean_absolute_error(y_te, np.exp(ridge_model.predict(X_te_l)))
### END CODE HERE ###
print(f"(a) raw: {mae_raw:.3f} | (b) log: {mae_log:.3f} | (c) log + ridge: {mae_log_ridge:.3f}")

In [ ]:
check_true(mae_log < mae_raw, "log-transform giảm MAE")

✍️ Giải thích:

**Gợi ý:** giá được sinh theo quan hệ **nhân** (diện tích^0.95 × hệ số quận × e^(−0.01·tuổi) × nhiễu lognormal). Trên thang log, quan hệ trở thành **cộng tuyến tính** và nhiễu có phương sai đều → model tuyến tính phù hợp hơn hẳn (giải quyết cả phi tuyến lẫn heteroscedasticity). Ridge với α nhỏ gần như không đổi gì vì $n \gg d$ và không có đa cộng tuyến nghiêm trọng.

## L6 — Thực tế

**Bài 9.** Xây pipeline dự đoán giá nhà **tốt nhất có thể** bằng model tuyến tính (được dùng: feature engineering, đa thức, log, Ridge/Lasso, chọn hyperparameter bằng CV trên train). Mục tiêu: **MAE test < 0.6 tỷ**. Lưu model cuối vào `best_model`, dự đoán `pred_best` trên `X_te` (giá gốc).

In [ ]:
### START CODE HERE ### (≈ 10 dòng)
from sklearn.linear_model import RidgeCV


def engineer(X):
    X = X.copy()
    X["log_area"] = np.log(X["area_m2"])
    X["area_per_bedroom"] = X["area_m2"] / X["bedrooms"]
    return X.drop(columns=["area_m2"])


best_model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30))).fit(engineer(X_tr), np.log(y_tr))
pred_best = np.exp(best_model.predict(engineer(X_te)))
### END CODE HERE ###
print(f"MAE test = {mean_absolute_error(y_te, pred_best):.3f} tỷ")

In [ ]:
check_less(mean_absolute_error(y_te, pred_best), 0.6, "MAE test < 0.6 tỷ")

## L7 — Nâng cao

**Bài 10.** Cài đặt **Lasso bằng coordinate descent** from scratch: `lasso_cd(X, y, lam, n_iters)` với X đã chuẩn hoá và y đã trừ trung bình, mục tiêu $\frac{1}{2n}\lVert y - Xw\rVert^2 + \lambda\lVert w\rVert_1$. Mỗi bước cập nhật $w_j$:

$$\rho_j = \frac1n X_{:,j}^\top\left(y - X w + X_{:,j}w_j\right), \qquad w_j = \frac{S(\rho_j, \lambda)}{\frac1n\lVert X_{:,j}\rVert^2}, \qquad S(\rho, \lambda) = \operatorname{sign}(\rho)\max(|\rho| - \lambda, 0)$$

So sánh với `sklearn.linear_model.Lasso(alpha=lam, fit_intercept=False)`.

In [ ]:
def soft_threshold(rho, lam):
    ### START CODE HERE ### (≈ 1 dòng)
    s = np.sign(rho) * max(abs(rho) - lam, 0.0)
    ### END CODE HERE ###
    return s


def lasso_cd(X, y, lam, n_iters=200):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(n_iters):
        for j in range(d):
            ### START CODE HERE ### (≈ 2 dòng)
            rho = X[:, j] @ (y - X @ w + X[:, j] * w[j]) / n
            w[j] = soft_threshold(rho, lam) / (X[:, j] @ X[:, j] / n)
            ### END CODE HERE ###
    return w

In [ ]:
Xl = StandardScaler().fit_transform(rng.normal(size=(200, 8)))
yl = Xl @ np.array([3.0, 0, 0, -2.0, 0, 0, 1.0, 0]) + rng.normal(0, 0.5, 200)
yl = yl - yl.mean()
w_ours = lasso_cd(Xl, yl, lam=0.2)
w_sk = Lasso(alpha=0.2, fit_intercept=False).fit(Xl, yl).coef_
check_close(w_ours, w_sk, tol=1e-4, message="khớp sklearn Lasso")
check_equal(int(np.sum(np.abs(w_ours) > 1e-8)), 3, "chọn đúng 3 feature có ích")